# Robustness v2 — seed-aware confidence intervals

**What this notebook fixes.** In `robustness_analysis.ipynb`, CELL 6 and CELL 8 built their 95% CIs by resampling **test items only**,
always keeping the same 5 trained models. That captures test-set luck (small) but ignores **seed luck** (large for Full-FT),
so the CIs came out too narrow.

This notebook recomputes every LoRA − Full calibration gap with:

| Column | Method | Captures |
|---|---|---|
| `item_lo / item_hi` | **old method** (items only) — kept for comparison | test-set luck only |
| `hier_lo / hier_hi` | **hierarchical bootstrap**: resample seeds, then items within each seed | seed luck + test-set luck |
| `t_lo / t_hi` | seed-level paired t-interval (n = 5, t = 2.776) | seed luck |

**Decision rule:** a gap is called `ROBUST` only if **both** the hierarchical CI **and** the t CI lie entirely on one side of 0.

**How to run:** from the repository root, `pip install numpy pandas scipy jupyter`, then
`jupyter nbconvert --to notebook --execute analysis/robustness_v2.ipynb` (or open it and *Run all*).
It reads `results/preds/` and writes to `results/robustness/`. CPU only, a few minutes.
Set the `THESIS_ROOT` environment variable to point it at another folder holding `preds/`.

In [ ]:
# CELL 0 - imports and configuration
import os, glob, warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar

# ROOT is the folder that holds preds/. Default: this repository's results/ folder,
# found whether the notebook runs from the repo root or from analysis/.
ROOT = os.environ.get('THESIS_ROOT') or next(
    (p for p in ['results', os.path.join('..', 'results')] if os.path.isdir(os.path.join(p, 'preds'))), None)
assert ROOT, 'results/preds not found: run from the repo root or set THESIS_ROOT'

PREDS_DIR = os.path.join(ROOT, 'preds')
OUT_DIR   = os.path.join(ROOT, 'robustness')
os.makedirs(OUT_DIR, exist_ok=True)

FRACTIONS = [0.01, 0.05, 0.10]
SEEDS     = [7, 13, 21, 33, 42]
ARMS      = ['full', 'lora']
N_BOOT    = int(os.environ.get('N_BOOT', 2000))   # bootstrap resamples per CI
RNG_SEED  = 20260910      # fixed so results are reproducible
warnings.filterwarnings('ignore')
print('ROOT =', ROOT)
print('Prediction files found:', len(glob.glob(os.path.join(PREDS_DIR, '*.npz'))), '(expected 30)')

In [ ]:
# CELL 1 - load test-set predictions 
npz_files = sorted(glob.glob(os.path.join(PREDS_DIR, '*.npz')))
LOGIT_KEYS = ['logits', 'logit', 'test_logits', 'preds', 'y_logits', 'scores']
LABEL_KEYS = ['labels', 'label', 'y_true', 'gold', 'targets', 'y']

def _detect_keys(path):
    d = np.load(path, allow_pickle=True); keys = list(d.files)
    lk = next((k for k in LOGIT_KEYS if k in keys), None)
    yk = next((k for k in LABEL_KEYS if k in keys), None)
    return lk, yk

def _leading_number(text):
    num = ''
    for ch in text:
        if ch.isdigit() or ch == '.': num += ch
        else: break
    return num

def parse_run_name(fname):
    base = os.path.basename(fname).lower()
    arm = 'full' if (base.startswith('full') or ('full' in base and 'lora' not in base)) else ('lora' if 'lora' in base else None)
    frac = None
    if 'frac' in base:
        num = _leading_number(base.split('frac', 1)[1]).rstrip('.')
        if num: frac = min(FRACTIONS, key=lambda f: abs(f - float(num)))
    seed = None
    if 'seed' in base:
        tail = base.split('seed', 1)[1]
        digs = ''
        for ch in tail.lstrip('_-'):
            if ch.isdigit(): digs += ch
            else: break
        if digs: seed = int(digs)
    return arm, frac, seed

assert npz_files, 'No .npz files found - check ROOT in CELL 0.'
LK, YK = _detect_keys(npz_files[0])
print('logits key =', LK, '| labels key =', YK)

PRED = {}
for p in npz_files:
    arm, frac, seed = parse_run_name(p)
    if None in (arm, frac, seed):
        print('  could not parse:', os.path.basename(p)); continue
    d = np.load(p, allow_pickle=True)
    PRED[(arm, frac, seed)] = {'logits': np.asarray(d[LK], dtype=np.float64),
                               'labels': np.asarray(d[YK]).astype(int).ravel()}
print('Loaded', len(PRED), 'runs (expected 30).')

for frac in FRACTIONS:
    for s in SEEDS:
        kf, kl = ('full', frac, s), ('lora', frac, s)
        assert kf in PRED and kl in PRED, f'missing run {kf} or {kl}'
        assert np.array_equal(PRED[kf]['labels'], PRED[kl]['labels']), f'test labels differ at {frac}, seed {s}'
print('OK: every (fraction, seed) has both arms on an identical test set.')

In [ ]:
# CELL 2 - metrics 
def softmax(logits):
    z = logits - logits.max(axis=1, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(labels, probs, eps=1e-12):
    p = np.clip(probs[np.arange(len(labels)), labels], eps, 1.0)
    return float(-np.log(p).mean())

def brier_multiclass(labels, probs):
    onehot = np.eye(probs.shape[1])[labels]
    return float(((probs - onehot) ** 2).sum(1).mean())

def ece_equal_width(labels, probs, n_bins=15):
    # primary ECE: bins span the attainable range [1/C, 1]
    conf = probs.max(1); correct = (probs.argmax(1) == labels).astype(float)
    C = probs.shape[1]
    edges = np.linspace(1.0 / C, 1.0, n_bins + 1); edges[0] = 0.0; edges[-1] = 1.0 + 1e-9
    N = len(labels); ece = 0.0
    for i in range(n_bins):
        m = (conf >= edges[i]) & (conf < edges[i + 1])
        if m.sum() == 0: continue
        ece += (m.sum() / N) * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def ece_adaptive(labels, probs, n_bins=15):
    conf = probs.max(1); correct = (probs.argmax(1) == labels).astype(float)
    order = np.argsort(conf); conf_s, corr_s = conf[order], correct[order]
    N = len(labels); edges = np.linspace(0, N, n_bins + 1).astype(int); ece = 0.0
    for i in range(n_bins):
        s, e = edges[i], edges[i + 1]
        if e <= s: continue
        ece += ((e - s) / N) * abs(corr_s[s:e].mean() - conf_s[s:e].mean())
    return float(ece)

def fit_temperature(logits, labels):
    res = minimize_scalar(lambda T: nll(labels, softmax(logits / T)), bounds=(0.05, 10.0), method='bounded')
    return float(res.x)

METRICS = [('ECE15',       lambda y, p: ece_equal_width(y, p, 15)),
           ('ECE_adapt15', lambda y, p: ece_adaptive(y, p, 15)),
           ('Brier',       brier_multiclass),
           ('NLL',         nll)]

PROBS = {k: softmax(v['logits']) for k, v in PRED.items()}
print('Metrics ready.')

In [ ]:
# CELL 3 - sanity check: per-seed values must match results/results.csv
rows = []
for frac in FRACTIONS:
    for s in SEEDS:
        y = PRED[('full', frac, s)]['labels']
        r = dict(fraction=frac, seed=s)
        for arm in ARMS:
            r[f'ece15_{arm}'] = ece_equal_width(y, PROBS[(arm, frac, s)], 15)
        r['diff_lora_minus_full'] = r['ece15_lora'] - r['ece15_full']
        rows.append(r)
SANITY = pd.DataFrame(rows)
print(SANITY.round(4).to_string(index=False))

# Cross-check against src/analyze.py: mean ECE per arm, as printed in results/stats.txt
EXPECTED = {(0.01, 'full'): 0.2446, (0.01, 'lora'): 0.1167,
            (0.05, 'full'): 0.1026, (0.05, 'lora'): 0.0791,
            (0.10, 'full'): 0.1202, (0.10, 'lora'): 0.0530}
for (frac, arm), want in EXPECTED.items():
    got = SANITY[SANITY.fraction == frac][f'ece15_{arm}'].mean()
    assert abs(got - want) < 5e-5, f'ECE mismatch at {frac} {arm}: {got:.4f} vs stats.txt {want:.4f}'
print('OK: per-arm mean ECE matches results/stats.txt (src/analyze.py).')

## The core function

For each fraction and metric, this computes the per-seed paired differences (LoRA − Full) and three 95% CIs.

- **Item-only (old):** every resample keeps all 5 seeds exactly once and only resamples test items.
- **Hierarchical (new):** level 1 draws 5 seeds *with replacement*; level 2 resamples test items inside each drawn seed.
- **Seed-level t:** mean ± 2.776 × SD/√5 over the 5 per-seed differences.

In [ ]:
# CELL 4 - item-only vs hierarchical vs t-interval
def gap_cis(frac, metric_fn, prob_source, n_boot=N_BOOT, rng=None):
    rng = rng if rng is not None else np.random.default_rng(RNG_SEED)
    pairs = {s: (prob_source[('full', frac, s)], prob_source[('lora', frac, s)], PRED[('full', frac, s)]['labels'])
             for s in SEEDS}
    d = np.array([metric_fn(y, pl) - metric_fn(y, pf) for pf, pl, y in pairs.values()])
    N = len(next(iter(pairs.values()))[2])

    item_boot, hier_boot = [], []
    for _ in range(n_boot):
        # OLD: same 5 seeds every time, resample items only
        idx = rng.integers(0, N, N)
        item_boot.append(np.mean([metric_fn(y[idx], pl[idx]) - metric_fn(y[idx], pf[idx])
                                  for pf, pl, y in pairs.values()]))
        # NEW: resample seeds, then items inside each drawn seed
        drawn = rng.choice(SEEDS, len(SEEDS), replace=True)
        diffs = []
        for s in drawn:
            pf, pl, y = pairs[s]
            j = rng.integers(0, N, N)
            diffs.append(metric_fn(y[j], pl[j]) - metric_fn(y[j], pf[j]))
        hier_boot.append(np.mean(diffs))

    n = len(d); se = d.std(ddof=1) / np.sqrt(n); tcrit = stats.t.ppf(0.975, n - 1)
    item_lo, item_hi = np.percentile(item_boot, [2.5, 97.5])
    hier_lo, hier_hi = np.percentile(hier_boot, [2.5, 97.5])
    t_lo, t_hi = d.mean() - tcrit * se, d.mean() + tcrit * se
    excl = lambda lo, hi: (hi < 0) or (lo > 0)
    return dict(mean_diff=d.mean(), sd_diff=d.std(ddof=1),
                lora_wins=f'{int((d < 0).sum())}/{n}',
                item_lo=item_lo, item_hi=item_hi,
                hier_lo=hier_lo, hier_hi=hier_hi,
                t_lo=t_lo, t_hi=t_hi,
                old_verdict='excludes 0' if excl(item_lo, item_hi) else 'includes 0',
                verdict='ROBUST' if (excl(hier_lo, hier_hi) and excl(t_lo, t_hi)) else 'NOT ROBUST',
                per_seed_diffs=np.round(d, 4).tolist())

rows = []
for frac in FRACTIONS:
    for name, fn in METRICS:
        r = gap_cis(frac, fn, PROBS)
        rows.append(dict(fraction=frac, metric=name, **r))
        print(f'frac {frac:.2f} {name:11s} diff={r["mean_diff"]:+.4f}  '
              f'old[{r["item_lo"]:+.4f},{r["item_hi"]:+.4f}]  '
              f'hier[{r["hier_lo"]:+.4f},{r["hier_hi"]:+.4f}]  '
              f't[{r["t_lo"]:+.4f},{r["t_hi"]:+.4f}]  wins {r["lora_wins"]}  -> {r["verdict"]}')
CALIB_CI = pd.DataFrame(rows)
CALIB_CI.to_csv(os.path.join(OUT_DIR, 'calibration_diff_ci_v2.csv'), index=False)
print()
print('Saved -> results/robustness/calibration_diff_ci_v2.csv')
print('Negative diff = LoRA lower (better). ROBUST = hierarchical AND t CI both exclude 0.')

In [ ]:
# CELL 5 - temperature scaling, re-done with seed-aware CIs (replaces old CELL 8 CIs)
# Same protocol as the old CELL 8: T fitted by 5-fold cross-validation on the test set, cv_seed = run seed.
def cv_temperature_calibrate(logits, labels, k_folds=5, cv_seed=0):
    N = len(labels); order = np.random.default_rng(cv_seed).permutation(N)
    folds = np.array_split(order, k_folds)
    cal = np.zeros((N, logits.shape[1])); Ts = []
    for i in range(k_folds):
        te = folds[i]; tr = np.concatenate([folds[j] for j in range(k_folds) if j != i])
        T = fit_temperature(logits[tr], labels[tr]); Ts.append(T)
        cal[te] = softmax(logits[te] / T)
    return cal, float(np.mean(Ts))

CALIB = {}
for k, v in PRED.items():
    CALIB[k], _ = cv_temperature_calibrate(v['logits'], v['labels'], 5, cv_seed=k[2])

ece_fn = METRICS[0][1]
rows = []
for frac in FRACTIONS:
    for stage, src in [('before', PROBS), ('after', CALIB)]:
        r = gap_cis(frac, ece_fn, src)
        rows.append(dict(fraction=frac, stage=stage, **r))
        print(f'frac {frac:.2f} {stage:6s} ECE gap={r["mean_diff"]:+.4f}  '
              f'old[{r["item_lo"]:+.4f},{r["item_hi"]:+.4f}]  '
              f'hier[{r["hier_lo"]:+.4f},{r["hier_hi"]:+.4f}]  '
              f't[{r["t_lo"]:+.4f},{r["t_hi"]:+.4f}]  wins {r["lora_wins"]}  -> {r["verdict"]}')
TEMP_CI = pd.DataFrame(rows)
TEMP_CI.to_csv(os.path.join(OUT_DIR, 'temp_scaling_gap_v2.csv'), index=False)
print()
print('Saved -> results/robustness/temp_scaling_gap_v2.csv')

In [ ]:
# CELL 6 - stability: how much each arm's calibration varies across seeds
rows = []
for frac in FRACTIONS:
    for name, fn in METRICS:
        for arm in ARMS:
            vals = [fn(PRED[(arm, frac, s)]['labels'], PROBS[(arm, frac, s)]) for s in SEEDS]
            rows.append(dict(fraction=frac, metric=name, arm=arm,
                             mean=np.mean(vals), sd=np.std(vals, ddof=1),
                             min=np.min(vals), max=np.max(vals)))
STAB = pd.DataFrame(rows)
wide = STAB.pivot_table(index=['fraction', 'metric'], columns='arm', values='sd')
wide['sd_ratio_full_over_lora'] = wide['full'] / wide['lora']
print('Across-seed SD of each metric (higher = less stable):')
print(wide.round(4).to_string())
STAB.to_csv(os.path.join(OUT_DIR, 'seed_stability_v2.csv'), index=False)
print()
print('Saved -> results/robustness/seed_stability_v2.csv')
print('Note: SD ratios from n=5 are themselves noisy; report them descriptively, not as a test.')

In [ ]:
# CELL 7 - plain-language summary (generated from the numbers above; check it against the tables)
print('=== What you may claim (ECE15, raw probabilities) ===')
for frac in FRACTIONS:
    r = CALIB_CI[(CALIB_CI.fraction == frac) & (CALIB_CI.metric == 'ECE15')].iloc[0]
    if r.verdict == 'ROBUST' and r.mean_diff < 0:
        msg = 'LoRA robustly lower ECE (both CIs below 0).'
    elif r.verdict == 'ROBUST':
        msg = 'Full-FT robustly lower ECE (both CIs above 0).'
    else:
        msg = 'No robust difference (a CI includes 0) - do NOT claim LoRA is better calibrated here.'
    flip = '  [changed vs old notebook]' if (r.old_verdict == 'excludes 0') != (r.verdict == 'ROBUST') else ''
    print(f'  {frac:.2f}: diff {r.mean_diff:+.4f}, LoRA wins {r.lora_wins} -> {msg}{flip}')

print()
print('=== After temperature scaling ===')
for frac in FRACTIONS:
    r = TEMP_CI[(TEMP_CI.fraction == frac) & (TEMP_CI.stage == 'after')].iloc[0]
    msg = 'residual LoRA advantage is ROBUST' if (r.verdict == 'ROBUST' and r.mean_diff < 0) else 'no robust residual gap'
    print(f'  {frac:.2f}: diff {r.mean_diff:+.4f}, LoRA wins {r.lora_wins} -> {msg}')

print()
print('Methods sentence (edit to match your thesis):')
print('  "Confidence intervals for LoRA - full fine-tuning differences were obtained by a two-level')
print(f'   (hierarchical) bootstrap ({N_BOOT} resamples) that resamples training seeds and then test items')
print('   within each seed, and were cross-checked with a seed-level paired t-interval (n = 5).')
print('   A difference is reported as robust only when both intervals exclude zero."')